If you are new to PheTK, please check out the APOE PheWAS - Logistic Regression notebook first. 

This notebook demonstrates how to run the same PheWAS pipeline in command line interface (CLI).

NOTE: certain CLI features, e.g., progress bar, might not work properly when running CLI commands in Jupyter Notebook. In practice, users should run CLI version in a native CLI environment, e.g., in the terminal.

## Initial setup

In [ ]:
!pip install --upgrade \
--index-url https://test.pypi.org/simple/ \
--extra-index-url https://pypi.org/simple/ \
phetk==0.2.1rc134

The above cell output was cleared to avoid exposing sensitive information.

In [2]:
!pip show PheTK | grep Version

Version: 0.2.1rc134


__Please restart the notebook kernel before proceeding to the next step.__

In [1]:
from phetk.cohort import Cohort
from phetk.phecode import Phecode
from phetk.phewas import PheWAS

## PheWAS with PheTK in CLI

In [2]:
%%bash
# exit script if a command fails
set -e

# add phetk path
export PATH=$PATH:/home/jupyter/.local/bin

# create cohort
echo "Creating cohort..."
echo
phetk cohort by-genotype \
--platform aou \
--aou_db_version 8 \
--chromosome_number 19 \
--genomic_position 44908684 \
--ref_allele T \
--alt_allele C \
--gt_dict '{"0": "0/0", "1":["0/1", "1/1"]}' \
--output_file_path "rs429358_cohort.tsv"

# add covariates
echo "Adding covariates..."
echo
phetk cohort add-covariates \
--cohort_file_path rs429358_cohort.tsv \
--age_at_last_ehr_event true \
--sex_at_birth true \
--first_n_pcs 5 \
--drop_nulls true \
--output_file_path rs429358_cohort_with_covariates.tsv

# generate phecode counts
echo "Generating phecode counts..."
echo
phetk phecode count-phecode \
--platform aou \
--phecode_version X \
--output_file_path aou_phecode_counts.tsv

# run PheWAS
echo "Running PheWAS..."
echo
phetk phewas \
--phecode_version X \
--phecode_count_file_path aou_phecode_counts.tsv \
--cohort_file_path rs429358_cohort_with_covariates.tsv \
--sex_at_birth_col sex_at_birth \
--male_as_one true \
--covariate_cols age_at_last_ehr_event sex_at_birth pc1 pc2 pc3 pc4 pc5 \
--independent_variable_of_interest genotype \
--min_cases 50 \
--min_phecode_count 2 \
--method logit \
--output_file_path rs429358_phewas_results.tsv

Creating cohort...



/opt/conda/lib/python3.10/site-packages/hail/context.py:350: UserWarning: Using hl.init with a default_reference argument is deprecated. To set a default reference genome after initializing hail, call `hl.default_reference` with an argument to set the default reference genome.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/hailtop/aiocloud/aiogoogle/user_config.py:43: UserWarning: Reading spark-defaults.conf to determine GCS requester pays configuration. This is deprecated. Please use `hailctl config set gcs_requester_pays/project` and `hailctl config set gcs_requester_pays/buckets`.
  warnings.warn(
Running on Apache Spark version 3.5.3
SparkUI available at http://all-of-us-5797-m.us-central1-c.c.terra-vpc-sc-7883f2cd.internal:34783
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.134-952ae203dbbe
LOGGING: writing to /home/jupyter/workspaces/phetkv8/hail-20250801-0323-0.2.134-952ae203dbbe.log
2025-08-01 03:24:18.414 Ha


Locus chr19:44908684 found!
+----------------+------------+----------+---------+-----------+
| locus          | alleles    | filters  | a_index | was_split |
+----------------+------------+----------+---------+-----------+
| locus<GRCh38>  | array<str> | set<str> |   int32 |      bool |
+----------------+------------+----------+---------+-----------+
| chr19:44908684 | ["T","C"]  | {}       |       1 |     False |
+----------------+------------+----------+---------+-----------+

+--------------------------+---------------------------+
| variant_qc.gq_stats.mean | variant_qc.gq_stats.stdev |
+--------------------------+---------------------------+
|                  float64 |                   float64 |
+--------------------------+---------------------------+
|                 4.30e+01 |                  8.93e+00 |
+--------------------------+---------------------------+

+-------------------------+-------------------------+----------------------+
| variant_qc.gq_stats.min | variant_qc

100%|██████████| 42/42 [02:19<00:00,  3.32s/it]



Cohort size: 280444 participants
Genotype 0: 204525 participants
Genotype 1: 75919 participants

Cohort data saved as rs429358_cohort_with_covariates.tsv

Generating phecode counts...

Start querying ICD codes...
Done!

Mapping ICD codes to phecodeX...
Successfully generated phecodeX counts for cohort participants!

Saved to aou_phecode_counts.tsv

Running PheWAS...

~~~~~~~~~~~~~~~~~~~~~~~~    Creating PheWAS Instance    ~~~~~~~~~~~~~~~~~~~~~~~~

Cohort size:  280444
genotype descriptions:  shape: (2, 2)
┌──────────┬────────┐
│ genotype ┆ count  │
│ ---      ┆ ---    │
│ i64      ┆ u32    │
╞══════════╪════════╡
│ 1        ┆ 75919  │
│ 0        ┆ 204525 │
└──────────┴────────┘

Number of unique phecodes in cohort:  3430
Total number of phecode events:  20812932
Number of phecode batches to process:  3430

Analysis method:  Logistic regression

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~    Running PheWAS    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Parallelization method: multithreading
Number of workers: 11

...

Multithreading completed successfully.
Combining 2517 result files...


Reading files: 100%|██████████| 2517/2517 [00:03<00:00, 789.47it/s]


Concatenating results...
Cleaning up temporary files...


Cleaning files: 100%|██████████| 2517/2517 [00:00<00:00, 34665.49it/s]



~~~~~~~~~~~~~~~~~~~~~~~~~~~~    PheWAS Completed    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~

Number of participants in cohort: 280444
Number of phecodes in cohort: 3430
Number of phecodes having less than 50 cases or controls: 913
Number of phecodes tested: 2517
Suggested Bonferroni correction (-log₁₀ scale): 4.701913211212344
Number of phecodes above Bonferroni correction: 16

PheWAS results saved to rs429358_phewas_results.tsv 

